# Colorizar una foto familiar en blanco y negro

Este cuaderno construye el MVP paso a paso: una persona de la familia carga una foto, un modelo de difusión propone colores, CLIP compara la imagen antes y después y Gradio muestra ambas versiones.

> **Importante:** una foto en blanco y negro no contiene sus colores originales. El modelo genera una propuesta verosímil, no recupera ni verifica la verdad histórica. Conservá el original.

El trabajo usa un ControlNet de Hugging Face y un modelo base Stable Diffusion 2.1. La ficha de ControlNet solo declara licencia OpenRAIL y casi no documenta sus datos o resultados; lo probaremos como experimento, no como una solución validada. Sus pesos ocupan alrededor de 1,46 GB, además de los varios GB del modelo base.

## Antes de empezar

- Seleccioná el kernel **Python 3.12 (colorizador-fotos)**; usa el único entorno `.venv` del proyecto.
- Hace falta Internet para descargar los modelos la primera vez.
- Los modelos seleccionados son públicos y no requieren autenticación. Sus licencias OpenRAIL igualmente tienen condiciones de uso.
- Esta computadora tiene PyTorch CPU, sin CUDA. La inferencia puede tardar mucho; para una GPU se puede adaptar el entorno en Colab. No subas fotos familiares privadas a un entorno remoto sin permiso.

Recorrido: rutas y dispositivo → modelos → preparación de imagen → colorización → comparación y CLIP → interfaz local → exportación opcional de `app.py`.

## 1. Preparar las carpetas

Guardamos cachés y temporales dentro del proyecto en E:. Así se evita llenar la unidad del sistema. Ejecutá esta celda antes de importar Diffusers o Transformers.

In [2]:
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path(r"E:\ifts24-lab-pdi-2026\TP2")

CACHE_DIR = ROOT / ".model-cache"
HF_HOME = CACHE_DIR / "huggingface"
OUTPUT_DIR = CACHE_DIR / "resultados"
for carpeta in (HF_HOME, OUTPUT_DIR, CACHE_DIR / "torch", CACHE_DIR / "gradio"):
    carpeta.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["GRADIO_TEMP_DIR"] = str(CACHE_DIR / "gradio")
os.environ["TEMP"] = str(CACHE_DIR)
os.environ["TMP"] = str(CACHE_DIR)

print("Proyecto:", ROOT)
print("Caché de modelos:", HF_HOME)
print("Resultados:", OUTPUT_DIR)

Proyecto: e:\ifts24-lab-pdi-2026\TP2
Caché de modelos: e:\ifts24-lab-pdi-2026\TP2\.model-cache\huggingface
Resultados: e:\ifts24-lab-pdi-2026\TP2\.model-cache\resultados


## 2. Comprobar el entorno

Elegimos automáticamente GPU si PyTorch la detecta; si no, usamos CPU. En CPU se conserva precisión completa. No se fuerza una descarga ni se ejecuta todavía el modelo.

In [3]:
import torch
from PIL import Image, ImageOps
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: la difusión en CPU puede tardar mucho; usá una foto pequeña para la primera prueba.")

PyTorch: 2.15.0+cpu
Dispositivo: cpu
Aviso: la difusión en CPU puede tardar mucho; usá una foto pequeña para la primera prueba.


## 3. Modelos públicos de Hugging Face

El modelo base [`sd2-community/stable-diffusion-2-1-base`](https://huggingface.co/sd2-community/stable-diffusion-2-1-base) y el ControlNet de colorización son públicos en Hugging Face: no hace falta iniciar sesión ni ingresar un token. Ambos publican sus pesos bajo licencias OpenRAIL; revisá sus condiciones de uso. El repositorio base informa la arquitectura Stable Diffusion 2.1, compatible con el ControlNet elegido.

## 4. Cargar el modelo de difusión

ControlNet recibe la estructura de la foto en escala de grises y condiciona la generación. El modelo base propone los colores. Fijamos las revisiones de ambos repositorios para que la descarga sea reproducible. La primera descarga del modelo base ocupa varios GB.

In [4]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

CONTROLNET_ID = "neurallove/controlnet-sd21-colorization-diffusers"
CONTROLNET_REVISION = "2083978ca9345c9edac3af2779c17710b7e42258"
BASE_MODEL_ID = "sd2-community/stable-diffusion-2-1-base"
BASE_MODEL_REVISION = "4e63672c03103b6c636b8fb4119ba982469b2955"

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID,
    revision=CONTROLNET_REVISION,
    use_safetensors=False,
    torch_dtype=DTYPE,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    controlnet=controlnet,
    torch_dtype=DTYPE,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=False)

PIPELINE_LISTO = True
print("Modelo de difusión cargado en", DEVICE)

e:\ifts24-lab-pdi-2026\TP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 6/6 [00:20<00:00,  3.45s/it]


Modelo de difusión cargado en cpu


## 5. Preparar una imagen sin deformarla

Convertimos la entrada a blanco y negro para que el experimento sea consistente. Reducimos el lado mayor a 512 píxeles y redondeamos dimensiones a múltiplos de 8, que son adecuados para la arquitectura de Stable Diffusion. El resultado se ampliará a las dimensiones originales para poder compararlo.

In [5]:
def preparar_imagen(imagen, lado_maximo=512):
    """Devuelve la foto gris y su versión reducida para el modelo."""
    original = ImageOps.grayscale(imagen).convert("RGB")
    ancho, alto = original.size
    escala = min(1.0, lado_maximo / max(ancho, alto))
    nuevo_ancho = max(64, int(ancho * escala) // 8 * 8)
    nuevo_alto = max(64, int(alto * escala) // 8 * 8)
    reducida = original.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    return original, reducida

print("Función de preparación lista. La interfaz permitirá cargar una imagen cuando esté completa.")

Función de preparación lista. La interfaz permitirá cargar una imagen cuando esté completa.


## 6. Generar la propuesta de color

La semilla hace que una repetición con los mismos parámetros produzca un resultado más comparable. El prompt orienta el estilo; no aporta información verdadera sobre los colores de la escena. Bajá los pasos para una prueba rápida, sabiendo que eso también puede cambiar el resultado.

In [6]:
PROMPT = "colorized vintage black and white family photograph, natural realistic colors, preserve the original composition and details"
NEGATIVE_PROMPT = "black and white, grayscale, sepia, changed composition, distorted face, artifacts, text"

def colorizar(imagen, semilla=42, pasos=15):
    if "pipe" not in globals():
        raise RuntimeError("Primero ejecutá la celda que carga el modelo de difusión.")
    original, control = preparar_imagen(imagen)
    generador = torch.Generator(device=DEVICE).manual_seed(int(semilla))
    resultado = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=control,
        width=control.width,
        height=control.height,
        num_inference_steps=int(pasos),
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        generator=generador,
    ).images[0]
    resultado = resultado.resize(original.size, Image.Resampling.LANCZOS)
    return original, resultado

print("Función de colorización lista; todavía no se descargó ni ejecutó ninguna foto.")

Función de colorización lista; todavía no se descargó ni ejecutó ninguna foto.


## 7. Preparar el análisis visual con CLIP

CLIP compara la imagen con dos descripciones de texto. Sus puntajes expresan afinidad relativa con esas etiquetas: **no** miden calidad, naturalidad, identidad ni autenticidad de los colores. Cargamos este segundo modelo desde Hugging Face; también descarga sus propios pesos.

In [7]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()
CLIP_LISTO = True
print("CLIP cargado en", DEVICE)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


CLIP cargado en cpu


In [8]:
ETIQUETAS_CLIP = ["a black and white photograph", "a color photograph"]

@torch.inference_mode()
def analizar_con_clip(imagenes):
    entradas = clip_processor(
        text=ETIQUETAS_CLIP,
        images=imagenes,
        return_tensors="pt",
        padding=True,
    )
    entradas = {nombre: tensor.to(DEVICE) for nombre, tensor in entradas.items()}
    puntajes = clip_model(**entradas).logits_per_image.softmax(dim=1).cpu().tolist()
    return puntajes

print("Analizador CLIP listo.")

Analizador CLIP listo.


## 8. Preparar la comparación

Esta función reúne procesamiento y análisis para que la use tanto el cuaderno como Gradio. En la tabla veremos qué etiqueta se parece más a cada imagen; el puntaje no sirve como verificación del color real.

In [9]:
PRUEBA_OK = False

def procesar_foto(imagen, semilla=42, pasos=15):
    global PRUEBA_OK
    if imagen is None:
        raise ValueError("Cargá una imagen para comenzar.")
    original, colorizada = colorizar(imagen, semilla=semilla, pasos=pasos)
    puntajes = analizar_con_clip([original, colorizada])
    nombres = ["Original", "Colorizada"]
    reporte = "| Imagen | Blanco y negro | A color |\n|---|---:|---:|\n"
    for nombre, fila in zip(nombres, puntajes):
        reporte += f"| {nombre} | {fila[0]:.1%} | {fila[1]:.1%} |\n"
    reporte += "\n*CLIP estima afinidad con dos etiquetas; no comprueba los colores originales.*"
    PRUEBA_OK = True
    return original, colorizada, reporte

print("Flujo integrado listo. La primera prueba real se hará al cargar una foto en la interfaz.")

Flujo integrado listo. La primera prueba real se hará al cargar una foto en la interfaz.


## 9. Abrir la interfaz desde este cuaderno

La app corre localmente desde el kernel. `share=False` evita crear un enlace público. Cargá una foto no sensible para la primera prueba; la comparación y el aviso aparecen en la misma pantalla.

In [ ]:
import gradio as gr

with gr.Blocks(title="Colorizador de fotos familiares") as demo:
    gr.Markdown(
        "# Colorizador de fotos familiares\n"
        "**Los colores son una propuesta de IA, no los colores históricos comprobados. Conservá tu original.**"
    )
    with gr.Row():
        entrada = gr.Image(label="Foto en blanco y negro", type="pil")
        original_salida = gr.Image(label="Original en escala de grises", type="pil")
        color_salida = gr.Image(label="Propuesta colorizada", type="pil")
    pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión (más pasos pueden tardar más)")
    semilla = gr.Number(value=42, precision=0, label="Semilla")
    boton = gr.Button("Colorizar y analizar")
    analisis = gr.Markdown()
    boton.click(
        fn=procesar_foto,
        inputs=[entrada, semilla, pasos],
        outputs=[original_salida, color_salida, analisis],
    )

demo.launch(share=False, inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


100%|██████████| 15/15 [2:32:34<00:00, 610.32s/it] 


In [10]:
import gradio as gr

with gr.Blocks(title="Colorizador de fotos familiares") as demo:
    gr.Markdown(
        "# Colorizador de fotos familiares"   
    )
    with gr.Row():
        entrada = gr.Image(label="Foto en blanco y negro", type="pil")
        original_salida = gr.Image(label="Original, escala de grises", type="pil")
        color_salida = gr.Image(label="Propuesta colorizada", type="pil")
    pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión (más pasos pueden tardar más)")
    semilla = gr.Number(value=42, precision=0, label="Semilla")
    boton = gr.Button("Colorizar y analizar")
    analisis = gr.Markdown()
    boton.click(
        fn=procesar_foto,
        inputs=[entrada, semilla, pasos],
        outputs=[original_salida, color_salida, analisis],
    )

demo.launch(share=False, inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


100%|██████████| 8/8 [2:43:20<00:00, 1225.04s/it]  


## 10. Qué observar y explicar

1. ¿Qué formas y tonos de la foto original se conservaron?
2. ¿Qué zonas recibieron colores que no podemos comprobar?
3. ¿Cambió CLIP su afinidad hacia “a color”? Eso describe la etiqueta, no la calidad.
4. ¿Qué pasa al repetir con otra semilla o cantidad de pasos?

La foto no se envía a una API de pago ni se publica mediante un enlace compartido. El modelo agrega contenido plausible y puede alterar detalles; compará siempre con el original.

## 11. Exportar `app.py` después de una prueba correcta

Ejecutá esta celda solo después de cargar ambos modelos y completar al menos una colorización desde la interfaz. Genera un script autocontenido con el mismo flujo y un `main guard`; no lo inicia. El cuaderno sigue siendo la forma de probar la app durante el desarrollo.

In [10]:
if not globals().get("PIPELINE_LISTO", False) or not globals().get("CLIP_LISTO", False):
    raise RuntimeError("Cargá y verificá primero los modelos de difusión y CLIP.")
if not globals().get("PRUEBA_OK", False):
    raise RuntimeError("Primero procesá una foto correctamente desde la interfaz.")

APP_SOURCE = r'''from pathlib import Path
import os

ROOT = Path(__file__).resolve().parent
CACHE_DIR = ROOT / ".model-cache"
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["GRADIO_TEMP_DIR"] = str(CACHE_DIR / "gradio")
os.environ["TEMP"] = str(CACHE_DIR)
os.environ["TMP"] = str(CACHE_DIR)

import gradio as gr
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from PIL import Image, ImageOps
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
CONTROLNET_ID = "neurallove/controlnet-sd21-colorization-diffusers"
CONTROLNET_REVISION = "2083978ca9345c9edac3af2779c17710b7e42258"
BASE_MODEL_ID = "sd2-community/stable-diffusion-2-1-base"
BASE_MODEL_REVISION = "4e63672c03103b6c636b8fb4119ba982469b2955"
CLIP_ID = "openai/clip-vit-base-patch32"
PROMPT = "colorized vintage black and white family photograph, natural realistic colors, preserve the original composition and details"
NEGATIVE_PROMPT = "black and white, grayscale, sepia, changed composition, distorted face, artifacts, text"
ETIQUETAS_CLIP = ["a black and white photograph", "a color photograph"]

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID, revision=CONTROLNET_REVISION, use_safetensors=False, torch_dtype=DTYPE
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL_ID, revision=BASE_MODEL_REVISION, controlnet=controlnet, torch_dtype=DTYPE
).to(DEVICE)
pipe.enable_attention_slicing()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()

def preparar_imagen(imagen, lado_maximo=512):
    original = ImageOps.grayscale(imagen).convert("RGB")
    ancho, alto = original.size
    escala = min(1.0, lado_maximo / max(ancho, alto))
    nuevo_ancho = max(64, int(ancho * escala) // 8 * 8)
    nuevo_alto = max(64, int(alto * escala) // 8 * 8)
    reducida = original.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    return original, reducida

def colorizar(imagen, semilla=42, pasos=15):
    original, control = preparar_imagen(imagen)
    generador = torch.Generator(device=DEVICE).manual_seed(int(semilla))
    resultado = pipe(
        prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, image=control,
        width=control.width, height=control.height,
        num_inference_steps=int(pasos), guidance_scale=7.5,
        controlnet_conditioning_scale=1.0, generator=generador,
    ).images[0]
    return original, resultado.resize(original.size, Image.Resampling.LANCZOS)

@torch.inference_mode()
def analizar_con_clip(imagenes):
    entradas = clip_processor(
        text=ETIQUETAS_CLIP, images=imagenes, return_tensors="pt", padding=True
    )
    entradas = {nombre: tensor.to(DEVICE) for nombre, tensor in entradas.items()}
    return clip_model(**entradas).logits_per_image.softmax(dim=1).cpu().tolist()

def procesar_foto(imagen, semilla=42, pasos=15):
    if imagen is None:
        raise gr.Error("Cargá una imagen para comenzar.")
    original, colorizada = colorizar(imagen, semilla=semilla, pasos=pasos)
    puntajes = analizar_con_clip([original, colorizada])
    reporte = "| Imagen | Blanco y negro | A color |\n|---|---:|---:|\n"
    for nombre, fila in zip(["Original", "Colorizada"], puntajes):
        reporte += f"| {nombre} | {fila[0]:.1%} | {fila[1]:.1%} |\n"
    reporte += "\n*CLIP compara etiquetas; no comprueba los colores originales.*"
    return original, colorizada, reporte

def crear_interfaz():
    with gr.Blocks(title="Colorizador de fotos familiares") as demo:
        gr.Markdown("# Colorizador de fotos familiares\n**Los colores son una propuesta de IA. Conservá tu original.**")
        with gr.Row():
            entrada = gr.Image(label="Foto en blanco y negro", type="pil")
            original_salida = gr.Image(label="Original en escala de grises", type="pil")
            color_salida = gr.Image(label="Propuesta colorizada", type="pil")
        pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión")
        semilla = gr.Number(value=42, precision=0, label="Semilla")
        boton = gr.Button("Colorizar y analizar")
        analisis = gr.Markdown()
        boton.click(procesar_foto, [entrada, semilla, pasos], [original_salida, color_salida, analisis])
    return demo

if __name__ == "__main__":
    crear_interfaz().launch(share=False)
'''

app_path = ROOT / "app.py"
app_path.write_text(APP_SOURCE, encoding="utf-8")
print("Generado:", app_path)

Generado: e:\ifts24-lab-pdi-2026\TP2\app.py
